In [1]:
%matplotlib inline
import sys
import os
import time
import numpy as np

import matplotlib.pyplot as plt
import matplotlib as mpl

# Parameters

In [2]:
# parameters
def a_m(m,b): return (8*np.sqrt(2)/np.pi) * ((m**2)/(4*m**2 - 1)) * ((b**2 + m**2 -1)/(b**2 + m**2))

def b_m(m,b,beta): return (beta*b**2)/(b**2 + m**2)

def d_m(m,b): return (64*np.sqrt(2)/(15*np.pi)) * ((b**2 - m**2 +1)/(b**2 + m**2))
    
def g_m_tilde(m,b,gamma): return gamma * ((4*m)/(4*m**2 -1)) * (b*np.sqrt(2)/np.pi)

e = 16*np.sqrt(2)/(5*np.pi)

def g_m(m,b,gamma): return gamma * ((4*m**3)/(4*m**2 -1)) * (b*np.sqrt(2)/(np.pi*(b**2 + m**2)))

In [4]:
x_1_star = 0.95
x_4_star = -0.76095
C = 0.1 
gamma = 0.2
beta = 1.25
b = 1.6
s = 0.05

a_1, a_2, b_1, b_2 = a_m(1,b), a_m(2,b), b_m(1,b,beta), b_m(2,b,beta)
d_1, d_2 = d_m(1,b), d_m(2,b)
g_1_tilde, g_2_tilde = g_m_tilde(1,b,gamma), g_m_tilde(2,b,gamma)
g_1, g_2 = g_m(1,b,gamma), g_m(2,b,gamma)
F1 = C*x_1_star
F4 = C*x_4_star

# Model 1 trajectory

In [5]:
def CdV_model(x_0, F1, F4, C, a_1, a_2, b_1, b_2, d_1, d_2, g_1_tilde, g_2_tilde, g_1, g_2, e, s, dt, N, rand_seed = 123):
    
    #np.random.seed(rand_seed)
    
    # Create a local random generator (this way I do not affect the random seed globally)
    rng = np.random.default_rng(rand_seed)
    
    ### Inputs
    # x_0: initial condition. E.g. np.array([0.01,0.2,0.3])
    # F: forcing
    # L: linear operator.
    # B: nonlinear operator
    # S: variance of the Wiener process
    # dt: integration time stepping
    # N: number of integration steps
    
    # Return a long trajectory with N time steps
    
    # Array for results
    x = np.zeros([N+1,len(x_0)])
    
    # Set initial condition
    x[0] = x_0
    
    # Defined Wiener process
    wiener = rng.normal(0, 1, (N,len(x_0)))*np.sqrt(dt)
    
    # forward integration via Euler-Mayurama scheme
    for t in range(N):
        
        x[t+1,0] = x[t,0] + (F1 - C*x[t,0] + g_1_tilde*x[t,2])*dt + s*wiener[t,0]
        x[t+1,1] = x[t,1] + (-C*x[t,1] + b_1*x[t,2] - a_1*x[t,0]*x[t,2] -d_1*x[t,3]*x[t,5])*dt + s*wiener[t,1]
        x[t+1,2] = x[t,2] + (-C*x[t,2] - b_1*x[t,1] + a_1*x[t,0]*x[t,1] + d_1*x[t,3]*x[t,4] -g_1*x[t,0])*dt + s*wiener[t,2]
        x[t+1,3] = x[t,3] + (F4 -C*x[t,3] + e*x[t,1]*x[t,5] - e*x[t,2]*x[t,4] + g_2_tilde*x[t,5])*dt + s*wiener[t,3]
        x[t+1,4] = x[t,4] + (-C*x[t,4] + b_2*x[t,5] - a_2*x[t,0]*x[t,5] -d_2*x[t,3]*x[t,2])*dt + s*wiener[t,4]
        x[t+1,5] = x[t,5] + (-C*x[t,5] - b_2*x[t,4] +a_2*x[t,0]*x[t,4] + d_2*x[t,3]*x[t,1] - g_2*x[t,3])*dt + s*wiener[t,5]

    return x

In [7]:
########## Test

N = 100000+10000 # The first 1000 are a transient
dt = 0.01
x_0 = np.random.normal(0,1,6)
orbit = CdV_model(x_0, F1, F4, C, a_1, a_2, b_1, b_2, 
                  d_1, d_2, g_1_tilde, g_2_tilde, 
                  g_1, g_2, e, s, dt, N, rand_seed = 123)[10000:]

# "Parallel" implementation

In [5]:
def CvD_model_parallel(x_0, 
                         F1, F4, C, 
                         a_1, a_2, 
                         b_1, b_2, 
                         d_1, d_2, 
                         g_1_tilde, g_2_tilde, 
                         g_1, g_2, e, s,
                    dt,N,rand_seed = 123):
    
    #np.random.seed(rand_seed)
    
    # Create a local random generator (this way I do not affect the random seed globally)
    rng = np.random.default_rng(rand_seed)
    
    ### Inputs
    # x_0: ensemble of initial condition. E.g. for N_ens = 2, np.array([[0.01,0.2,0.3],[0.1,0.52,1.3]])
    # F: forcing
    # L: linear operator.
    # B: nonlinear operator
    # S: variance of the Wiener process
    # dt: integration time stepping
    # N: number of integration steps
    
    # Return the last integration points for the whole ensemble
    
    # Number of ensemble members
    Nens = x_0.shape[0]
    
    # Set initial condition
    x = x_0.copy() # Use copy so it does not overwrite it
    
    # Initialize the ensemble of trajectories
    ensemble = np.zeros([Nens,N,6])
    
    # forward integration via Euler-Mayurama scheme
    for t in range(N):
        
        # Defined Wiener process
        wiener = rng.normal(0, 1, (Nens,6))*np.sqrt(dt)
        
        x[:,0] = x[:,0] + (F1 - C*x[:,0] + g_1_tilde*x[:,2])*dt + s*wiener[:,0]
        x[:,1] = x[:,1] + (-C*x[:,1] + b_1*x[:,2] - a_1*x[:,0]*x[:,2] -d_1*x[:,3]*x[:,5])*dt + s*wiener[:,1]
        x[:,2] = x[:,2] + (-C*x[:,2] - b_1*x[:,1] + a_1*x[:,0]*x[:,1] + d_1*x[:,3]*x[:,4] -g_1*x[:,0])*dt + s*wiener[:,2]
        x[:,3] = x[:,3] + (F4 -C*x[:,3] + e*x[:,1]*x[:,5] - e*x[:,2]*x[:,4] + g_2_tilde*x[:,5])*dt + s*wiener[:,3]
        x[:,4] = x[:,4] + (-C*x[:,4] + b_2*x[:,5] - a_2*x[:,0]*x[:,5] -d_2*x[:,3]*x[:,2])*dt + s*wiener[:,4]
        x[:,5] = x[:,5] + (-C*x[:,5] - b_2*x[:,4] +a_2*x[:,0]*x[:,4] + d_2*x[:,3]*x[:,1] - g_2*x[:,3])*dt + s*wiener[:,5]
        
        ensemble[:,t] = x

    return ensemble

In [8]:
%%time

########## Test

# Number of ensemble members
N_ens = 1000

N = 1000 # The first 1000 are a transient
dt = 0.01

x_0 = np.random.normal(0,1,(N_ens,6))

#print(str(x_0))

# I want to check that if I run two ensembles than we are going 
# to have the same results if we fix the noise

ensemble_a = CvD_model_parallel(x_0, 
                         F1, F4, C, 
                         a_1, a_2, 
                         b_1, b_2, 
                         d_1, d_2, 
                         g_1_tilde, g_2_tilde, 
                         g_1, g_2, e, s,
                    dt,N,rand_seed = 123)

#print(str(x_0))

ensemble_b = CvD_model_parallel(x_0, 
                         F1, F4, C, 
                         a_1, a_2, 
                         b_1, b_2, 
                         d_1, d_2, 
                         g_1_tilde, g_2_tilde, 
                         g_1, g_2, e, s,
                    dt,N,rand_seed = 123)

# Yes, they are the same, so all good.

CPU times: user 574 ms, sys: 40.9 ms, total: 615 ms
Wall time: 616 ms


# Averages on the fly: Parallel implementation

In [5]:
from scipy.stats import skew
from scipy.stats import kurtosis

### Unforced ensemble
def CvD_model_parallel_avgOnTheFly(x_0, 
                                   F1, F4, C, 
                                   a_1, a_2, 
                                   b_1, b_2, 
                                   d_1, d_2, 
                                   g_1_tilde, g_2_tilde, 
                                   g_1, g_2, e, s,
                                   dt, N, rand_seed=123):

    rng = np.random.default_rng(rand_seed)
    Nens = x_0.shape[0]
    x = x_0.copy()
    
    avg_x = np.zeros([N, 6])
    var_x = np.zeros([N, 6])
    
    for t in range(N):
        avg_x[t] = np.mean(x, axis=0)
        var_x[t] = np.var(x, axis=0)
        
        wiener = rng.normal(0, 1, (Nens, 6)) * np.sqrt(dt)
        
        x_prev = x.copy()  # snapshot for simultaneous updates

        x[:,0] = x_prev[:,0] + (F1 - C*x_prev[:,0] + g_1_tilde*x_prev[:,2])*dt + s*wiener[:,0]
        x[:,1] = x_prev[:,1] + (-C*x_prev[:,1] + b_1*x_prev[:,2] - a_1*x_prev[:,0]*x_prev[:,2] - d_1*x_prev[:,3]*x_prev[:,5])*dt + s*wiener[:,1]
        x[:,2] = x_prev[:,2] + (-C*x_prev[:,2] - b_1*x_prev[:,1] + a_1*x_prev[:,0]*x_prev[:,1] + d_1*x_prev[:,3]*x_prev[:,4] - g_1*x_prev[:,0])*dt + s*wiener[:,2]
        x[:,3] = x_prev[:,3] + (F4 - C*x_prev[:,3] + e*x_prev[:,1]*x_prev[:,5] - e*x_prev[:,2]*x_prev[:,4] + g_2_tilde*x_prev[:,5])*dt + s*wiener[:,3]
        x[:,4] = x_prev[:,4] + (-C*x_prev[:,4] + b_2*x_prev[:,5] - a_2*x_prev[:,0]*x_prev[:,5] - d_2*x_prev[:,3]*x_prev[:,2])*dt + s*wiener[:,4]
        x[:,5] = x_prev[:,5] + (-C*x_prev[:,5] - b_2*x_prev[:,4] + a_2*x_prev[:,0]*x_prev[:,4] + d_2*x_prev[:,3]*x_prev[:,1] - g_2*x_prev[:,3])*dt + s*wiener[:,5]

    return avg_x, var_x

# Impulse response functions

In [6]:
x_1_star = 0.95
x_4_star = -0.76095
C = 0.01 
gamma = 0.2
beta = 1.25
b = 1.6
s = 0.05

a_1, a_2, b_1, b_2 = a_m(1,b), a_m(2,b), b_m(1,b,beta), b_m(2,b,beta)
d_1, d_2 = d_m(1,b), d_m(2,b)
g_1_tilde, g_2_tilde = g_m_tilde(1,b,gamma), g_m_tilde(2,b,gamma)
g_1, g_2 = g_m(1,b,gamma), g_m(2,b,gamma)
F1 = C*x_1_star
F4 = C*x_4_star

## Compute changes of mean and variance to impulse perturbations

In [ ]:
%%time

period_long = 4000 # This is Model Time Unit (MTI) = 50
    
# Number of ensemble members
N_ens = 10**6 #10000000
dt = 0.01

# Load the long trajectory simulated above
sigmas = np.load('./results/sigma_invariant_density.npy')

for i in range(6):

    eps = 0.1*sigmas[i]
    if i == 0:
        delta = np.array([eps,0,0,0,0,0])
    elif i == 1:
        delta = np.array([0,eps,0,0,0,0])
    elif i == 2:
        delta = np.array([0,0,eps,0,0,0])
    elif i == 3:
        delta = np.array([0,0,0,eps,0,0])
    elif i == 4:
        delta = np.array([0,0,0,0,eps,0])
    elif i == 5:
        delta = np.array([0,0,0,0,0,eps])    
    
    print('perturbation = '+str(delta))
    
    '''
    Let us sample initial conditions directly from the attractor. 
    To do so, we consider a long integration of the system and random sample 
    an ensemble of initial conditions from it
    '''
    
    # Load the long trajectory, but keep only the first 10^7 points
    long_trajectory = np.load('./results/CdV_long_trajectory.npy')#[0:10**7]
    
    # Consider N_ens initial conditions
    # I have N_ens indices going from 0 to the length of the trajectory
    idx = np.random.randint(0, len(long_trajectory),N_ens)
    
    # We here considering these initial conditions equispaces
    #n_steps = long_trajectory.shape[0]
    # Equally spaced indices between 0 and n_steps-1
    #idx = np.linspace(0, n_steps - 1, N_ens, dtype=int)
    x_0 = long_trajectory[idx]
    
    ########## Ensemble of control simulations on the attractor
        
    # Initial condition
    init_cond = x_0
    
    # Generate a random seed (or set it manually for reproducibility)
    rand_seed_c = np.random.randint(0, 2**32 - 1)  # Random seed
    C_avg_x, C_var_x = CvD_model_parallel_avgOnTheFly(init_cond, 
                             F1, F4, C, 
                             a_1, a_2, 
                             b_1, b_2, 
                             d_1, d_2, 
                             g_1_tilde, g_2_tilde, 
                             g_1, g_2, e, s,
                        dt,period_long,rand_seed = rand_seed_c)
    
    ########## Ensemble of perturbed simulations on the attractor
    
    # Perturbed initial condition
    init_cond_p = x_0 + delta
    
    P_avg_x, P_var_x = CvD_model_parallel_avgOnTheFly(init_cond_p, 
                             F1, F4, C, 
                             a_1, a_2, 
                             b_1, b_2, 
                             d_1, d_2, 
                             g_1_tilde, g_2_tilde, 
                             g_1, g_2, e, s,
                        dt,period_long,rand_seed = rand_seed_c)
    
    # Note, I am using the same seed as the control run

    # Response in the mean
    R_mean = (P_avg_x-C_avg_x)/eps
    R_var = (P_var_x-C_var_x)/eps

    np.save('./results_responses/impulse_response_mean_x_'+str(i)+'.npy',R_mean)
    np.save('./results_responses/impulse_response_var_x_'+str(i)+'.npy',R_var)

perturbation = [0.02190969 0.         0.         0.         0.         0.        ]
perturbation = [0.         0.02657737 0.         0.         0.         0.        ]
perturbation = [0.         0.         0.02441791 0.         0.         0.        ]
perturbation = [0.         0.         0.         0.01693989 0.         0.        ]
perturbation = [0.         0.         0.         0.         0.01427531 0.        ]
perturbation = [0.         0.         0.         0.         0.         0.01331356]
